# Quantum-Enhanced RAG vs Classical RAG — NFCorpus

This notebook builds two retrieval pipelines on the same dataset (NFCorpus) so they can be fairly compared:

1. **Classical RAG**: Text → BGE-M3 embedding → FAISS cosine search
2. **Quantum RAG**: Text → BGE-M3 embedding → PCA (1024→4) → 4-Qubit Variational Quantum Circuit (VQC) → FAISS cosine search

The only difference between the two pipelines is the PCA + VQC step — everything else (documents, embedding model, chunking, distance metric) is kept identical, so that any difference in retrieval quality can be attributed to the quantum embedding step, not to some other confound.

**Evaluation metric**: Recall@5 — for a given question, did at least one of the top 5 retrieved documents match a document a human annotator actually marked as relevant (from NFCorpus's `qrels`)?


## Step 0 — Install dependencies

- `datasets` → load NFCorpus from HuggingFace
- `sentence-transformers` → run BGE-M3
- `faiss-cpu` → vector similarity search / indexing
- `pennylane` → build and train the quantum circuit (VQC)
- `scikit-learn` → PCA
- `torch` → training the VQC (PennyLane's torch interface plugs directly into PyTorch's autograd)


In [29]:
!pip install datasets sentence-transformers faiss-cpu pennylane scikit-learn numpy torch -q

## Step 1 — Load the NFCorpus dataset

NFCorpus is a benchmark retrieval dataset (medical/nutrition abstracts) with three separate pieces:

- **`corpus`**: the actual documents. Each has `_id`, `title`, `text`.
- **`queries`**: real questions. Each has `_id`, `text`.
- **`qrels`**: the human-verified answer key. Each row is `{query-id, corpus-id, score}` — `score > 0` means a human confirmed that document genuinely answers that question. This is what makes a real, quantitative retrieval evaluation possible (Recall@k, etc.) instead of just eyeballing results.

Corpus and queries are independent lists — `qrels` is the only thing that links a specific question to its specific correct answer(s).


In [30]:
from datasets import load_dataset

corpus = load_dataset("BeIR/nfcorpus", "corpus")["corpus"]
queries = load_dataset("BeIR/nfcorpus", "queries")["queries"]
qrels = load_dataset("BeIR/nfcorpus-qrels")["test"]

print(f"Corpus size: {len(corpus)}")
print(f"Queries size: {len(queries)}")
print(f"Qrels rows: {len(qrels)}")
print(corpus[0])
print(queries[0])
print(qrels[0])

Corpus size: 3633
Queries size: 3237
Qrels rows: 12334
{'_id': 'MED-10', 'title': 'Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland', 'text': 'Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of

## Step 2 — Chunking

NFCorpus documents are already short (abstract-length, a few hundred words), well within BGE-M3's 8192-token limit. Splitting an already-short abstract into smaller pieces would only fragment coherent ideas and add unnecessary bookkeeping (which chunk came from which document, for qrels matching).

**Decision: 1 document = 1 chunk, no splitting.** Title and text are concatenated, since the title usually carries strong topical signal that helps the embedding model.


In [31]:
def build_chunks(corpus):
    """
    For NFCorpus, each document = one chunk (title + text combined).
    Returns a list of dicts: {chunk_id, doc_id, text}
    """
    chunks = []
    for doc in corpus:
        combined_text = (doc["title"] + ". " + doc["text"]).strip()
        chunks.append({
            "chunk_id": doc["_id"],   # same as doc_id since 1 doc = 1 chunk
            "doc_id": doc["_id"],
            "text": combined_text
        })
    return chunks

chunks = build_chunks(corpus)
print(chunks[0])
print(f"Total chunks: {len(chunks)}")

{'chunk_id': 'MED-10', 'doc_id': 'MED-10', 'text': 'Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland. Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.

## Step 3 — Build shared lookup tables (once, up front)

Both `qrels` and `queries` only ever give you **IDs**, never the actual content directly matched up. Rather than rebuilding these lookups multiple times in different functions (a source of several bugs earlier in development), we build them **once, here, as standalone variables** that every later step can reuse:

- **`doc_id_to_chunk`**: document ID → the full chunk object (text, etc.)
- **`query_id_to_text`**: question ID → the actual question text
- **`query_to_pos_docs`**: question ID → list of document IDs a human confirmed are relevant to it (`score > 0` in qrels)
- **`all_doc_ids`**: every document ID that exists — used later to sample random "negative" (presumed irrelevant) documents
- **`query_ids_with_pos`**: only the questions that have at least one confirmed relevant document — you can only build training/eval examples from these


In [32]:
doc_id_to_chunk = {chunk["doc_id"]: chunk for chunk in chunks}
query_id_to_text = {q["_id"]: q["text"] for q in queries}

query_to_pos_docs = {}
for row in qrels:
    if row["score"] > 0:
        query_to_pos_docs.setdefault(row["query-id"], []).append(row["corpus-id"])

all_doc_ids = list(doc_id_to_chunk.keys())
query_ids_with_pos = list(query_to_pos_docs.keys())

print(f"doc_id_to_chunk: {len(doc_id_to_chunk)} entries")
print(f"query_id_to_text: {len(query_id_to_text)} entries")
print(f"query_to_pos_docs: {len(query_to_pos_docs)} queries with known relevant docs")
print(f"all_doc_ids: {len(all_doc_ids)} total documents")

doc_id_to_chunk: 3633 entries
query_id_to_text: 3237 entries
query_to_pos_docs: 323 queries with known relevant docs
all_doc_ids: 3633 total documents


## Step 4 — Embed all chunks with BGE-M3

BGE-M3 converts each chunk of text into a dense 1024-dimensional vector. `normalize_embeddings=True` L2-normalizes every vector to unit length — this matters because it lets cosine similarity be computed as a plain inner product later (inner product on normalized vectors = cosine similarity), which is what FAISS's `IndexFlatIP` expects.

This step is also the **classical embedding** that the classical RAG baseline will use directly (no PCA, no VQC) — so this cell's output (`embeddings`) feeds *both* pipelines.


In [33]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("BAAI/bge-m3")

texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"Embeddings shape: {embeddings.shape}")  # (num_chunks, 1024)

for i, chunk in enumerate(chunks):
    chunk["embedding"] = embeddings[i]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/114 [00:00<?, ?it/s]

Embeddings shape: (3633, 1024)


## Step 5 — PCA: reduce 1024 → 4 dimensions

The VQC has 4 qubits, so it needs exactly 4 numbers per chunk. PCA finds the 4 directions of greatest variance in the 1024-dim embeddings and projects every vector onto just those 4.

**Critical rule**: PCA is fit **once**, on the corpus, and the exact same fitted transform is reused for queries later — never refit on queries. Refitting would put queries and documents in two different, incomparable 4-D coordinate systems.

We also check `explained_variance_ratio_` — this tells us how much of the original signal survives this aggressive 256x compression. (In our own run, this came out to ~11.8% total variance retained — a real, worth-reporting limitation, not a bug.)


In [34]:
saved_pca_path = "/kaggle/input/qrag-embed-arch/pca_model.pkl"  # adjust to your actual input path
from sklearn.decomposition import PCA
import pickle

try:
    with open(saved_pca_path, "rb") as f:
        pca = pickle.load(f)
    reduced_embeddings = pca.transform(embeddings)
    print(f"Loaded saved PCA from {saved_pca_path}")
except FileNotFoundError:
    pca = PCA(n_components=4)
    reduced_embeddings = pca.fit_transform(embeddings)
    print("No saved PCA found — fit a new one.")

print(f"Reduced shape: {reduced_embeddings.shape}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Total variance retained: {pca.explained_variance_ratio_.sum():.2%}")

with open("/kaggle/working/pca_model.pkl", "wb") as f:
    pickle.dump(pca, f)

for i, chunk in enumerate(chunks):
    chunk["pca_embedding"] = reduced_embeddings[i]

No saved PCA found — fit a new one.
Reduced shape: (3633, 4)
Explained variance ratio per component: [0.03702867 0.03015598 0.02710178 0.02410303]
Total variance retained: 11.84%


In [35]:
import numpy as np

for i in range(4):
    col = reduced_embeddings[:, i]
    print(f"Component {i}:")
    print(f"  min={col.min():.4f}  max={col.max():.4f}")
    print(f"  1st pct={np.percentile(col, 1):.4f}  99th pct={np.percentile(col, 99):.4f}")
    print(f"  median={np.median(col):.4f}  std={col.std():.4f}")
    print()

print(f"Global max abs value (this is what max_scale currently uses): {np.max(np.abs(reduced_embeddings)):.4f}")

Component 0:
  min=-0.3175  max=0.3593
  1st pct=-0.2776  99th pct=0.2728
  median=0.0041  std=0.1382

Component 1:
  min=-0.3168  max=0.3788
  1st pct=-0.2495  99th pct=0.2904
  median=-0.0037  std=0.1248

Component 2:
  min=-0.3792  max=0.3555
  1st pct=-0.2750  99th pct=0.2557
  median=0.0038  std=0.1183

Component 3:
  min=-0.3759  max=0.3136
  1st pct=-0.2778  99th pct=0.2376
  median=0.0070  std=0.1115

Global max abs value (this is what max_scale currently uses): 0.3792


## Step 6 — Scale PCA outputs into angles, and cache for both chunks and queries

The VQC uses each of the 4 PCA numbers as a **rotation angle** for a qubit. Since rotation angles are periodic (2π = back to start), raw PCA values are rescaled into roughly the `-π` to `π` range using the largest absolute value seen across the whole corpus (`max_scale`).

We also embed **queries** here for the first time (BGE-M3 → the *same* fitted PCA → scale), and cache both chunk and query scaled vectors in dictionaries keyed by ID. This means the expensive BGE-M3/PCA work only happens once — the training loop (and later, retrieval) just does fast dictionary lookups instead of recomputing embeddings every time.


In [36]:
import torch

# Compute per-component clip bounds from the corpus (fit once, reuse for queries — same rule as PCA)
lower_bounds = np.percentile(reduced_embeddings, 1, axis=0)   # shape (4,)
upper_bounds = np.percentile(reduced_embeddings, 99, axis=0)  # shape (4,)

def scale_to_angles(vec, lower_bounds, upper_bounds):
    """Clip each of the 4 components to its own [1st, 99th] percentile range,
    then scale independently to roughly [-pi, pi]."""
    clipped = np.clip(vec, lower_bounds, upper_bounds)
    # map [lower, upper] -> [-pi, pi] per component
    scaled = np.pi * (2 * (clipped - lower_bounds) / (upper_bounds - lower_bounds) - 1)
    return scaled

# Cache: chunk_id -> scaled 4-number vector
chunk_id_to_scaled_vec = {}
for i, chunk in enumerate(chunks):
    scaled = scale_to_angles(reduced_embeddings[i], lower_bounds, upper_bounds)
    chunk_id_to_scaled_vec[chunk["chunk_id"]] = scaled

# Queries: same BGE-M3 model, same fitted PCA, same clip bounds (never refit on queries)
query_texts = [q["text"] for q in queries]
query_embeddings = model.encode(query_texts, normalize_embeddings=True)
query_reduced = pca.transform(query_embeddings)

query_id_to_scaled_vec = {}
for i, q in enumerate(queries):
    scaled = scale_to_angles(query_reduced[i], lower_bounds, upper_bounds)
    query_id_to_scaled_vec[q["_id"]] = scaled

print(f"Cached {len(chunk_id_to_scaled_vec)} chunk vectors and {len(query_id_to_scaled_vec)} query vectors")

Cached 3633 chunk vectors and 3237 query vectors


## Step 7 — Define the 4-Qubit Variational Quantum Circuit (VQC)

Picture 4 dials (qubits), each able to point in any direction. The circuit does three things, in order:

1. **Encoding** (`encode_data`): each of the 4 PCA numbers turns one dial (`qml.RY` = rotate qubit `i` by angle `x[i]`). One number, one dial — no relationship between them yet.
2. **Variational layer** (`variational_layer`): the dials get entangled (linked together) and each gets an extra nudge, controlled by trainable `weights`. This is the only part that changes during training.
3. **Measurement** (`qml.expval(qml.PauliZ(i))`): read the final position of each dial as a plain number between -1 and +1. Four numbers in, four numbers out — this is the "quantum embedding."

`interface="torch"` makes the circuit differentiable via PyTorch's autograd, so gradients can flow through it during training exactly like any normal PyTorch layer.


In [37]:
import pennylane as qml

n_qubits = 4
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)

def encode_data(x):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)

def variational_layer(weights):
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))

@qml.qnode(dev, interface="torch")
def quantum_circuit(x, weights):
    encode_data(x)
    variational_layer(weights)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

print("VQC defined.")

VQC defined.


## Step 8 — Build training triplets from qrels

The VQC has no built-in notion of "relevant" vs "irrelevant" — its weights start random. Training needs concrete examples to check itself against: for a given question, which document is *actually* correct, and which one (almost certainly) isn't.

Each **triplet** = one (question, correct document, random wrong document) bundle:

1. Pick a random question that has at least one known correct answer (`query_ids_with_pos`)
2. Pick one of its confirmed-relevant documents → the **positive**
3. Pick a random document from the whole corpus, making sure it isn't secretly a correct answer → the **negative**

2000 of these bundles form the training dataset — conceptually the same idea as a labeled dataset for any ML model, just shaped as groups of three instead of (input, label) pairs, since the goal is "pull these two together, push that one apart" rather than classification.


In [38]:
import random

def build_triplets(query_ids_with_pos, query_to_pos_docs, query_id_to_text,
                    doc_id_to_chunk, all_doc_ids, num_triplets=2000):
    triplets = []
    for _ in range(num_triplets):
        qid = random.choice(query_ids_with_pos)
        pos_doc_id = random.choice(query_to_pos_docs[qid])

        neg_doc_id = random.choice(all_doc_ids)
        while neg_doc_id in query_to_pos_docs[qid]:
            neg_doc_id = random.choice(all_doc_ids)

        if qid in query_id_to_text and pos_doc_id in doc_id_to_chunk and neg_doc_id in doc_id_to_chunk:
            triplets.append({
                "query_id": qid,
                "query_text": query_id_to_text[qid],
                "pos_chunk": doc_id_to_chunk[pos_doc_id],
                "neg_chunk": doc_id_to_chunk[neg_doc_id]
            })

    return triplets

triplets = build_triplets(query_ids_with_pos, query_to_pos_docs, query_id_to_text,
                           doc_id_to_chunk, all_doc_ids, num_triplets=2000)
print(f"Built {len(triplets)} triplets")
print(triplets[0]["query_text"])
print(triplets[0]["pos_chunk"]["text"][:100])
print(triplets[0]["neg_chunk"]["text"][:100])

Built 2000 triplets
Childhood Constipation and Cow’s Milk
The Role of Cow's Milk Allergy in Pediatric Chronic Constipation: A Randomized Clinical Trial. Objec
Global and regional mortality from 235 causes of death for 20 age groups in 1990 and 2010: a systema


## Step 9 — Initialize trainable weights and the optimizer

`qml.StronglyEntanglingLayers.shape(...)` doesn't compute anything about your data — it just tells you how many trainable numbers you need for 4 qubits across 2 layers (shape `(2, 4, 3)`). Those numbers are then randomly initialized as a **PyTorch tensor with `requires_grad=True`**, so PyTorch tracks how changing each of them affects the loss later.

`optimizer = torch.optim.Adam([weights], lr=0.01)` creates the mechanism that will actually apply the nudges to `weights` once training tells it which direction to move — `lr` (learning rate) controls how big each nudge is.


In [39]:
weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)
saved_weights_path = "/kaggle/input/qrag-embed-arch/vqc_weights.pt"  # adjust to your actual input path

try:
    weights = torch.load(saved_weights_path)
    weights.requires_grad_(True)
    weights_loaded_from_disk = True
    print(f"Loaded saved weights from {saved_weights_path}, shape {weights.shape}")
except FileNotFoundError:
    weights = torch.tensor(
        np.random.uniform(0, 2 * np.pi, size=weight_shape),
        dtype=torch.float64,
        requires_grad=True
    )
    weights_loaded_from_disk = False
    print("No saved weights found — initialized random weights.")

optimizer = torch.optim.Adam([weights], lr=0.01)
print(f"Weight shape: {weights.shape}, requires_grad: {weights.requires_grad}")

No saved weights found — initialized random weights.
Weight shape: torch.Size([2, 4, 3]), requires_grad: True


In [40]:
import torch

sample_chunk_ids = list(chunk_id_to_scaled_vec.keys())[:5]

print("Checking untrained circuit outputs for 5 sample documents:\n")
outputs = []
for cid in sample_chunk_ids:
    vec = torch.tensor(chunk_id_to_scaled_vec[cid], dtype=torch.float64)
    with torch.no_grad():
        out = torch.stack(quantum_circuit(vec, weights)).numpy()
    outputs.append(out)
    print(f"{cid}: scaled_input={chunk_id_to_scaled_vec[cid].round(3)}  ->  circuit_output={out.round(4)}")

# Quick numeric check: are these outputs actually spread apart, or nearly identical?
outputs = np.array(outputs)
print(f"\nStd dev across these 5 outputs, per dimension: {outputs.std(axis=0).round(4)}")

Checking untrained circuit outputs for 5 sample documents:

MED-10: scaled_input=[-0.264  1.414 -0.342  1.48 ]  ->  circuit_output=[ 0.4405  0.0992  0.1179 -0.0211]
MED-14: scaled_input=[ 0.113  1.068 -0.163  1.218]  ->  circuit_output=[ 0.3004  0.0442 -0.0202  0.1316]
MED-118: scaled_input=[-0.719 -0.553  1.034  2.253]  ->  circuit_output=[-0.1154 -0.0687  0.0017 -0.1485]
MED-301: scaled_input=[ 2.605  1.538  0.594 -0.076]  ->  circuit_output=[-0.218   0.1999  0.112   0.1313]
MED-306: scaled_input=[ 0.712  2.06  -1.596  2.102]  ->  circuit_output=[ 0.0231  0.0088 -0.0363 -0.1216]

Std dev across these 5 outputs, per dimension: [0.2484 0.09   0.0664 0.1197]


## Step 10 — Triplet loss

For one triplet, `dist_pos` = distance between the question's quantum embedding and its correct answer's (want this **small**). `dist_neg` = distance to the wrong answer's (want this **large**). If `dist_pos` is already smaller than `dist_neg` by at least `margin`, the loss clips to exactly 0 — nothing to fix. Otherwise, the loss is a positive number that tells the optimizer to adjust the weights.


In [41]:
def triplet_loss(anchor_emb, pos_emb, neg_emb, margin=0.5):
    dist_pos = torch.norm(anchor_emb - pos_emb)
    dist_neg = torch.norm(anchor_emb - neg_emb)
    loss = torch.clamp(dist_pos - dist_neg + margin, min=0)
    return loss

print("Loss function defined.")

Loss function defined.


## Step 11 — Train the VQC

For every triplet: look up its 3 pre-cached scaled vectors → run all three through the *same* dials using the *current* weights → compute the loss → let PyTorch work out which direction each weight should move (`loss.backward()`) → apply the nudge (`optimizer.step()`). One full pass through all 2000 triplets = one epoch; we repeat this 20 times.

**Watch the printed `avg loss` per epoch** — it should generally trend downward as training proceeds. A loss that plateaus well above 0 (as opposed to going to ~0) indicates the circuit only partially learned to separate relevant from irrelevant — worth reporting honestly rather than treated as a bug.

⚠️ This runs the circuit `2000 × 3 × 20 = 120,000` times total — expect this cell to take a while.


In [42]:
n_epochs = 20

if weights_loaded_from_disk:
    print("Skipping training — using pre-trained weights loaded from disk.")
else:
    for epoch in range(n_epochs):
        total_loss = 0.0
        for triplet in triplets:
            optimizer.zero_grad()

            anchor_vec = torch.tensor(query_id_to_scaled_vec[triplet["query_id"]], dtype=torch.float64)
            pos_vec = torch.tensor(chunk_id_to_scaled_vec[triplet["pos_chunk"]["chunk_id"]], dtype=torch.float64)
            neg_vec = torch.tensor(chunk_id_to_scaled_vec[triplet["neg_chunk"]["chunk_id"]], dtype=torch.float64)

            anchor_emb = torch.stack(quantum_circuit(anchor_vec, weights))
            pos_emb = torch.stack(quantum_circuit(pos_vec, weights))
            neg_emb = torch.stack(quantum_circuit(neg_vec, weights))

            loss = triplet_loss(anchor_emb, pos_emb, neg_emb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{n_epochs}, avg loss: {total_loss / len(triplets):.4f}")

Epoch 1/20, avg loss: 0.4886
Epoch 2/20, avg loss: 0.4711
Epoch 3/20, avg loss: 0.4678
Epoch 4/20, avg loss: 0.4695
Epoch 5/20, avg loss: 0.4691
Epoch 6/20, avg loss: 0.4700
Epoch 7/20, avg loss: 0.4706
Epoch 8/20, avg loss: 0.4718
Epoch 9/20, avg loss: 0.4705
Epoch 10/20, avg loss: 0.4705
Epoch 11/20, avg loss: 0.4688
Epoch 12/20, avg loss: 0.4689
Epoch 13/20, avg loss: 0.4686
Epoch 14/20, avg loss: 0.4682
Epoch 15/20, avg loss: 0.4669
Epoch 16/20, avg loss: 0.4682
Epoch 17/20, avg loss: 0.4675
Epoch 18/20, avg loss: 0.4672
Epoch 19/20, avg loss: 0.4670
Epoch 20/20, avg loss: 0.4674


## Step 12 — Save the trained weights

Kaggle sessions lose everything in memory once closed. Save the trained VQC weights (and re-confirm the PCA model is saved) to `/kaggle/working/` — then **click "Save Version" and wait for it to finish** before closing the session, since that's what actually persists these files on Kaggle's servers.


In [43]:
torch.save(weights, "/kaggle/working/vqc_weights.pt")
print("VQC weights saved!")

with open("/kaggle/working/pca_model.pkl", "wb") as f:
    pickle.dump(pca, f)
print("PCA model saved!")

# Sanity check: reload and confirm it matches
weights_check = torch.load("/kaggle/working/vqc_weights.pt")
print(weights_check.shape)
print(torch.equal(weights, weights_check))  # should print True

VQC weights saved!
PCA model saved!
torch.Size([2, 4, 3])
True


## Step 13 — Generate quantum embeddings for all documents and build the FAISS index

Run every chunk's cached, scaled PCA vector through the now-**trained** VQC once, producing its final quantum embedding. `torch.no_grad()` is used since we're only *using* the trained circuit now, not training it further — this is faster and uses less memory.

`IndexFlatIP` (inner product) on L2-normalized vectors is equivalent to cosine similarity — "Flat" means exact brute-force search, which is fast enough at this corpus size (3,633 documents, 4 dimensions) that no approximate index is needed.


In [44]:
import faiss

quantum_embeddings_list = []
chunk_ids_ordered = []

with torch.no_grad():
    for chunk in chunks:
        cid = chunk["chunk_id"]
        scaled_vec = torch.tensor(chunk_id_to_scaled_vec[cid], dtype=torch.float64)
        q_emb = quantum_circuit(scaled_vec, weights)
        q_emb = torch.stack(q_emb).numpy()

        quantum_embeddings_list.append(q_emb)
        chunk_ids_ordered.append(cid)

quantum_embeddings_matrix = np.array(quantum_embeddings_list).astype("float32")
print(f"Quantum embeddings matrix shape: {quantum_embeddings_matrix.shape}")

quantum_dimension = quantum_embeddings_matrix.shape[1]
quantum_index = faiss.IndexFlatIP(quantum_dimension)

faiss.normalize_L2(quantum_embeddings_matrix)
quantum_index.add(quantum_embeddings_matrix)

print(f"Total vectors in quantum FAISS index: {quantum_index.ntotal}")

Quantum embeddings matrix shape: (3633, 4)
Total vectors in quantum FAISS index: 3633


## Step 14 — Quantum search function

A real question must go through the **identical** pipeline documents went through: BGE-M3 → the same fitted PCA → the same scaling → the same trained VQC. Only then is it comparable to what's stored in the FAISS index. FAISS returns row indices, which get mapped back to actual `chunk_id`s via `chunk_ids_ordered`.


In [45]:
def quantum_search(query_text, top_k=5):
    query_emb = model.encode([query_text], normalize_embeddings=True)[0]
    query_reduced = pca.transform([query_emb])[0]
    query_scaled = np.pi * (query_reduced / max_scale)
    query_scaled_tensor = torch.tensor(query_scaled, dtype=torch.float64)

    with torch.no_grad():
        q_emb = quantum_circuit(query_scaled_tensor, weights)
        q_emb = torch.stack(q_emb).numpy().astype("float32").reshape(1, -1)

    faiss.normalize_L2(q_emb)
    similarities, indices = quantum_index.search(q_emb, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        chunk_id = chunk_ids_ordered[idx]
        chunk = doc_id_to_chunk[chunk_id]
        results.append({
            "rank": rank + 1,
            "chunk_id": chunk_id,
            "similarity": float(similarities[0][rank]),
            "text_preview": chunk["text"][:150]
        })
    return results

# Quick smoke test
test_result = quantum_search(queries[0]["text"], top_k=3)
for r in test_result:
    print(r["rank"], r["chunk_id"], round(r["similarity"], 4))

1 MED-880 0.9762
2 MED-4858 0.9669
3 MED-2967 0.9668


## Step 15 — Classical RAG baseline (for a fair comparison)

This is the missing piece needed to interpret the quantum results: a **classical** pipeline that uses the exact same documents, same BGE-M3 model, same chunking, and same cosine-similarity search — just **without** PCA or the VQC. This isolates the effect of the quantum embedding step specifically, rather than comparing two systems that differ in more than one way.


In [46]:
classical_dimension = embeddings.shape[1]  # 1024
classical_index = faiss.IndexFlatIP(classical_dimension)

classical_embeddings = embeddings.astype("float32").copy()
faiss.normalize_L2(classical_embeddings)
classical_index.add(classical_embeddings)

print(f"Classical FAISS index size: {classical_index.ntotal}")

def classical_search(query_text, top_k=5):
    query_emb = model.encode([query_text], normalize_embeddings=True).astype("float32")
    faiss.normalize_L2(query_emb)
    similarities, indices = classical_index.search(query_emb, top_k)
    return [chunks[idx]["chunk_id"] for idx in indices[0]]

# Quick smoke test
print(classical_search(queries[0]["text"], top_k=3))

Classical FAISS index size: 3633
['MED-2434', 'MED-2439', 'MED-5341']


## Step 16 — Evaluation: Recall@5, classical vs quantum, on the SAME fixed test set

**Recall@5**: out of a set of test questions, for what fraction did at least one of the top-5 retrieved documents match a document qrels confirms is actually relevant?

A fixed random seed (`random.seed(42)`) locks in one specific sample of 100 test questions (all guaranteed to have at least one known-correct answer, via `query_ids_with_pos`), so both pipelines are evaluated on **exactly the same questions** — this is what makes the final comparison valid rather than comparing two different random samples.


In [47]:
def evaluate_recall_at_k(search_fn, test_qids, k=5, is_quantum=False):
    hits = 0
    for qid in test_qids:
        query_text = query_id_to_text[qid]
        actual_relevant = set(query_to_pos_docs[qid])

        if is_quantum:
            results = search_fn(query_text, top_k=k)
            retrieved_ids = set(r["chunk_id"] for r in results)
        else:
            retrieved_ids = set(search_fn(query_text, top_k=k))

        if len(actual_relevant & retrieved_ids) > 0:
            hits += 1

    return hits / len(test_qids)

random.seed(42)
fixed_test_qids = random.sample(query_ids_with_pos, 100)

classical_recall_5 = evaluate_recall_at_k(classical_search, fixed_test_qids, k=5, is_quantum=False)
quantum_recall_5 = evaluate_recall_at_k(quantum_search, fixed_test_qids, k=5, is_quantum=True)

print(f"Classical Recall@5: {classical_recall_5:.2%}")
print(f"Quantum   Recall@5: {quantum_recall_5:.2%}")
print(f"Difference (quantum - classical): {(quantum_recall_5 - classical_recall_5):+.2%}")

Classical Recall@5: 64.00%
Quantum   Recall@5: 3.00%
Difference (quantum - classical): -61.00%


## Interpreting the result

Some context worth writing into your report, regardless of what numbers come out:

- **This measures ONE thing**: whether adding PCA + a trained 4-qubit VQC on top of BGE-M3 embeddings helps, hurts, or doesn't change retrieval quality on NFCorpus, using Recall@5.
- Two upstream factors are worth reporting alongside this result if quantum underperforms: (1) `pca.explained_variance_ratio_.sum()` — how much signal survived the 1024→4 compression, and (2) how far the final training loss got from 0 — how well the VQC learned to separate relevant from irrelevant during training. Both bound how good the quantum pipeline could possibly be, independent of the retrieval evaluation itself.
- If you want additional evidence beyond Recall@5, the same `evaluate_recall_at_k` function works for other values of `k` (e.g. `k=1`, `k=10`), and you could also try a middle baseline — classical BGE-M3 embeddings passed through PCA only (no VQC) — to isolate how much of any gap is due to PCA's compression alone versus the VQC's transformation.
